In [1]:
#!/usr/bin/env python3
"""plot_net_range.py — RQ3 network figures (range workload, SidePart DP vs random).

    python3 plot_net_range.py  CSV_DIR  OUT_DIR  [--rtts 1 10] [--rtt-bw 1000]

CSV_DIR holds net_range_rtt<R>_bw<B>.csv (one row per system x setting x seed).
For every (system, setting, RTT, bandwidth) the point is the mean over seeds,
error bars are +-1 standard deviation.

Writes to OUT_DIR, each as .png and .pdf:
  net_range_tput_vs_bw[_<setting>]     committed txn/s vs bandwidth (log x), one line per system x RTT
  net_range_lat_tput_bw[_<setting>]    committed txn/s vs median epoch latency, labels = bandwidth
  net_range_bytes_vs_bw[_<setting>]    network KB per admitted txn vs bandwidth
  net_range_shards_vs_bw[_<setting>]   shards contacted per read round (contacted_frac x k) vs bandwidth
  net_range_tput_vs_rtt[_<setting>]    committed txn/s vs RTT at bandwidth --rtt-bw
Each figure is written as four single panels, one per setting:
_unif_unif / _skew_matched / _unif_skewq / _skew_mismatched (for LaTeX
subfigures).  --grid additionally writes the 2x2 figure (no suffix).

In Jupyter: run this cell, then call  main(["CSV_DIR", "OUT_DIR"]).
"""
import argparse, csv, glob, os, re, statistics, sys
from collections import defaultdict

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [2]:
plt.rcParams.update({"font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8, "legend.fontsize": 7.5,
                     "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "pdf.fonttype": 42, "ps.fonttype": 42})

SETTINGS = [  # csv value, panel title, file suffix (2x2 order: row by row)
    ("uniform_data+uniform_queries",   "Uniform data, uniform queries",   "unif_unif"),
    ("skewed_data+matched_queries",    "Skewed data, matched queries",    "skew_matched"),
    ("uniform_data+skewed_queries",    "Uniform data, skewed queries",    "unif_skewq"),
    ("skewed_data+mismatched_queries", "Skewed data, mismatched queries", "skew_mismatched"),
]
# (system in csv, legend text, colour for the first RTT, colour for later RTTs, marker, open marker?)
SYSTEMS = [
    ("DePart", "SidePart, Alg. 2 ($\\epsilon$={eps})", "#0b4f9c", "#5b8fd0", "o", False),
    ("Random", "SidePart, $\\epsilon$=0: random placement", "#ff7f0e", "#ffad66", "s", True),
]
RTT_STYLE = ["-", "-.", ":", (0, (5, 2, 1, 2))]          # line style per RTT (first, second, ...)
RANDOM_STYLE = ["--", ":", "-.", (0, (1, 1))]
RTT_MARKER = ["o", "^", "D", "v"]

In [3]:
def bw_label(b):
    return f"{b / 1000:g}G" if b >= 1000 else f"{b:g}M"

In [4]:
def load(csv_dir):
    data = defaultdict(list)       # (system, setting, rtt, bw) -> list of row dicts (one per seed)
    files = glob.glob(os.path.join(csv_dir, "net_range_rtt*_bw*.csv"))
    files = [f for f in files if not f.endswith("_subepoch.csv")]
    if not files:   # zip unpacked one level deeper
        files = [f for f in glob.glob(os.path.join(csv_dir, "**", "net_range_rtt*_bw*.csv"), recursive=True)
                 if not f.endswith("_subepoch.csv")]
    eps = {}
    for f in files:
        m = re.search(r"net_range_rtt(\d+(?:\.\d+)?)_bw(\d+(?:\.\d+)?)\.csv$", f)
        if not m:
            continue
        rtt, bw = float(m.group(1)), float(m.group(2))
        with open(f) as fh:
            for r in csv.DictReader(fh):
                txns = float(r["txns"]) or 1.0
                k = float(r["k"])
                row = {
                    "tput": float(r["throughput_txn_s"]),
                    "lat": float(r["epoch_ms_median"]),
                    "kb": float(r["net_bytes"]) / txns / 1024.0,          # KB per admitted txn
                    "shards": float(r["contacted_frac"]) * k,             # shards contacted per round
                    "k": k,
                }
                data[(r["system"], r["setting"], rtt, bw)].append(row)
                eps.setdefault(r["system"], r["epsilon"])
    print(f"  {len(files)} csv files, {sum(len(v) for v in data.values())} rows")
    return data, eps

In [5]:
def ms(vals):
    return (statistics.fmean(vals), statistics.stdev(vals) if len(vals) > 1 else 0.0) if vals else (float("nan"), 0.0)

In [6]:
def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("csv_dir")
    ap.add_argument("out")
    ap.add_argument("--rtts", type=float, nargs="+", default=[1, 10],
                    help="RTTs (ms) drawn in the bandwidth figures (the csv may also contain 100)")
    ap.add_argument("--rtt-bw", type=float, default=1000, help="bandwidth (Mbit/s) of the throughput-vs-RTT figure")
    ap.add_argument("--rtt-all", action="store_true", help="tput-vs-RTT figure: use every RTT in the csv (log x)")
    ap.add_argument("--grid", action="store_true", help="also write the 2x2 figures (default: single panels only)")
    a = ap.parse_args(argv)
    data, eps = load(a.csv_dir)
    if not data:
        print("!! no net_range_rtt*_bw*.csv in", os.path.abspath(a.csv_dir)); return
    os.makedirs(a.out, exist_ok=True)
    print("  output ->", os.path.abspath(a.out))
    bws = sorted({k[3] for k in data})
    all_rtts = sorted({k[2] for k in data})
    rtts = [r for r in a.rtts if r in all_rtts]

    def agg(sysname, setting, rtt, bw, key):
        return ms([r[key] for r in data.get((sysname, setting, rtt, bw), [])])

    def label(sysname, base, rtt=None):
        txt = base.format(eps=eps.get(sysname, "?"))
        return txt + (f", RTT {rtt:g} ms" if rtt is not None else "")

    def style_axes(ax):
        ax.grid(True, alpha=0.3, linewidth=0.6)

    # ---- one panel -------------------------------------------------------
    def panel(ax, kind, setting, title):
        for si, (sysname, base, c0, c1, mk, hollow) in enumerate(SYSTEMS):
            if kind == "rtt":
                xs, ys, es = [], [], []
                for rtt in (all_rtts if a.rtt_all else rtts):
                    m, s = agg(sysname, setting, rtt, a.rtt_bw, "tput")
                    if m == m:
                        xs.append(rtt); ys.append(m); es.append(s)
                ax.errorbar(xs, ys, yerr=es, color=c0, marker=mk, markersize=6, capsize=2,
                            linestyle="-" if si == 0 else "--", linewidth=1.6,
                            markerfacecolor="white" if hollow else c0, label=label(sysname, base))
                continue
            for ri, rtt in enumerate(rtts):
                color = c0 if ri == 0 else c1
                ls = (RTT_STYLE if si == 0 else RANDOM_STYLE)[ri % 4]
                marker = mk if ri == 0 else RTT_MARKER[ri % 4]
                alpha = 1.0 if ri == 0 else 0.75
                face = "white" if hollow else color
                if kind == "lat":
                    pts = [(agg(sysname, setting, rtt, bw, "lat"), agg(sysname, setting, rtt, bw, "tput"), bw)
                           for bw in bws]
                    pts = [p for p in pts if p[0][0] == p[0][0]]
                    ax.errorbar([p[0][0] for p in pts], [p[1][0] for p in pts],
                                xerr=[p[0][1] for p in pts], yerr=[p[1][1] for p in pts],
                                color=color, linestyle=ls, marker=marker, markersize=6, capsize=2,
                                linewidth=1.6, alpha=alpha, markerfacecolor=face,
                                label=label(sysname, base, rtt))
                    for (lx, _), (ty, _), bw in pts:
                        ax.annotate(bw_label(bw), (lx, ty), textcoords="offset points", xytext=(2, 4),
                                    fontsize=7, color=color, alpha=alpha)
                    continue
                key = {"tput": "tput", "bytes": "kb", "shards": "shards"}[kind]
                xs, ys, es = [], [], []
                for bw in bws:
                    m, s = agg(sysname, setting, rtt, bw, key)
                    if m == m:
                        xs.append(bw); ys.append(m); es.append(s)
                ax.errorbar(xs, ys, yerr=es, color=color, linestyle=ls, marker=marker, markersize=6,
                            capsize=2, linewidth=1.6, alpha=alpha, markerfacecolor=face,
                            label=label(sysname, base, rtt))
        if kind == "shards":
            ks = [r["k"] for (s_, st, _, _), rows in data.items() if st == setting for r in rows]
            if ks:
                ax.axhline(max(ks), color="grey", linestyle=":", linewidth=1, label="all shards ($k$)")
                ax.set_ylim(0, max(ks) * 1.15)
        rt = ", ".join(f"{r:g}" for r in rtts)
        xl = {"tput": f"Bandwidth (Mbit/s), RTT = {rt} ms",
              "bytes": f"Bandwidth (Mbit/s), RTT = {rt} ms",
              "shards": f"Bandwidth (Mbit/s), RTT = {rt} ms",
              "lat": f"Epoch latency (ms), RTT = {rt} ms, labels = bandwidth",
              "rtt": f"Network RTT (ms), bandwidth = {a.rtt_bw:g} Mbit/s"}[kind]
        yl = {"tput": "Committed throughput (txn/s)", "lat": "Committed throughput (txn/s)",
              "rtt": "Committed throughput (txn/s)", "bytes": "Network KB / admitted txn",
              "shards": "Shards contacted per round"}[kind]
        if kind in ("tput", "bytes", "shards") or (kind == "rtt" and a.rtt_all):
            ax.set_xscale("log")
        if kind != "shards":
            ax.set_ylim(bottom=0)
        if kind == "lat":                      # room for the bandwidth labels above the top points
            top = max(statistics.fmean([r["tput"] for r in rows]) for rows in data.values())
            ax.set_ylim(0, top * 1.12)
        ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_title(title)
        style_axes(ax)

    names = {"tput": "net_range_tput_vs_bw", "lat": "net_range_lat_tput_bw", "bytes": "net_range_bytes_vs_bw",
             "shards": "net_range_shards_vs_bw", "rtt": "net_range_tput_vs_rtt"}

    def save(fig, name):
        for ext in ("png", "pdf"):
            fig.savefig(os.path.join(a.out, f"{name}.{ext}"), dpi=300, bbox_inches="tight")
        plt.close(fig)
        print("  wrote", name)

    for kind, base in names.items():
        # 2x2 figure (only with --grid)
        if a.grid:
            fig, axes = plt.subplots(2, 2, figsize=(7.0, 4.9), sharey=(kind != "shards"), sharex=(kind == "lat"))
            for ax, (setting, title, _) in zip(axes.flat, SETTINGS):
                panel(ax, kind, setting, title)
            for ax in axes[:, 1]:
                if kind != "shards":
                    ax.set_ylabel("")
            h, l = axes.flat[0].get_legend_handles_labels()
            fig.legend(h, l, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False, fontsize=8)
            fig.tight_layout()
            save(fig, base)
        # one file per setting
        for setting, title, suffix in SETTINGS:
            fig, ax = plt.subplots(figsize=(3.4, 2.45))
            panel(ax, kind, setting, title)
            h, l = ax.get_legend_handles_labels()
            fig.legend(h, l, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=1 if len(l) <= 3 else 2,
                       frameon=False, fontsize=6.5)
            fig.tight_layout()
            save(fig, f"{base}_{suffix}")


if __name__ == "__main__" and "ipykernel" not in sys.modules:
    main()

In [ ]:
main(["rq3_net_alg3_off100_out", "net_figs_rtt"])